In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data/raw/inspire")

operations = pd.read_csv(DATA_DIR / "operations.csv.gz")
vitals = pd.read_csv(DATA_DIR / "vitals.csv.gz")
labs = pd.read_csv(DATA_DIR / "labs.csv.gz")
diagnosis = pd.read_csv(DATA_DIR / "diagnosis.csv.gz")

In [8]:
from pathlib import Path

Path.cwd()

WindowsPath('c:/Users/Shivani12/Downloads/Masters/NYU/Projects/inspire-surgical-copilot/notebooks')

In [9]:
df = operations.copy()

In [10]:
df = df[
    [
        "op_id",
        "subject_id",
        "age",
        "sex",
        "height",
        "weight",
        "race",
        "asa",
        "department",
        "antype",
        "emop",
        "icd10_pcs",
        "opstart_time",
        "icuin_time",
        "icuout_time",
        "discharge_time",
        "inhosp_death_time",
    ]
].copy()

In [11]:
df["bmi"] = df["weight"] / (df["height"] / 100) ** 2

In [13]:
df[["opstart_time"]].info()
vitals[["chart_time"]].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130960 entries, 0 to 130959
Data columns (total 1 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   opstart_time  130948 non-null  float64
dtypes: float64(1)
memory usage: 1023.2 KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66029310 entries, 0 to 66029309
Data columns (total 1 columns):
 #   Column      Dtype
---  ------      -----
 0   chart_time  int64
dtypes: int64(1)
memory usage: 503.8 MB


In [14]:
hr = vitals[vitals["item_name"] == "hr"].copy()

In [15]:
hr.head()

,op_id,subject_id,chart_time,item_name,value
65,428563245,155496940,1925,hr,84.0
70,428563245,155496940,1930,hr,66.0
76,428563245,155496940,1935,hr,56.0
84,428563245,155496940,1940,hr,50.0
98,428563245,155496940,1945,hr,50.0


In [16]:
hr.columns.tolist()

['op_id', 'subject_id', 'chart_time', 'item_name', 'value']

In [17]:
sample_op = df.iloc[0]["op_id"]
sample_op

np.int64(484069807)

In [20]:
hr[hr["op_id"] == sample_op].sort_values("chart_time")


,op_id,subject_id,chart_time,item_name,value
64048549,484069807,178742874,1115,hr,80.0
64048556,484069807,178742874,1120,hr,84.0
64048572,484069807,178742874,1125,hr,100.0
64048588,484069807,178742874,1130,hr,88.0
64048602,484069807,178742874,1135,hr,78.0
64048615,484069807,178742874,1140,hr,74.0
64048628,484069807,178742874,1145,hr,72.0
64048641,484069807,178742874,1150,hr,74.0
64048654,484069807,178742874,1155,hr,78.0
64048667,484069807,178742874,1160,hr,72.0


In [21]:
df[df["op_id"] == sample_op][
    ["op_id", "opstart_time"]
]

,op_id,opstart_time
0,484069807,1140.0


In [22]:
def add_latest_vital(df, vitals_df, vital_name, output_column):
    """
    Add the latest measurement of a vital sign before surgery.
    """

    # Keep only one vital
    vital = vitals_df[vitals_df["item_name"] == vital_name].copy()

    # Keep only measurements before surgery
    vital = (
        vital.merge(
            df[["op_id", "opstart_time"]],
            on="op_id",
            how="inner"
        )
    )

    vital = vital[vital["chart_time"] <= vital["opstart_time"]]

    # Keep the latest measurement
    vital = (
        vital.sort_values("chart_time")
             .groupby("op_id")
             .last()[["value"]]
             .rename(columns={"value": output_column})
             .reset_index()
    )

    # Merge back
    return df.merge(vital, on="op_id", how="left")

In [23]:
df = add_latest_vital(
    df,
    vitals,
    "hr",
    "heart_rate"
)

In [24]:
df[
    [
        "op_id",
        "heart_rate"
    ]
].head()

,op_id,heart_rate
0,484069807,74.0
1,446270725,74.0
2,406892271,92.0
3,478413008,92.0
4,468516791,78.0


In [25]:
df["heart_rate"].isna().mean()

np.float64(0.0024816737935247405)

In [26]:
df = add_latest_vital(df, vitals, "nibp_sbp", "systolic_bp")
df = add_latest_vital(df, vitals, "nibp_dbp", "diastolic_bp")
df = add_latest_vital(df, vitals, "rr", "respiratory_rate")
df = add_latest_vital(df, vitals, "spo2", "oxygen_saturation")
df = add_latest_vital(df, vitals, "bt", "temperature")

In [27]:
df[
    [
        "heart_rate",
        "systolic_bp",
        "diastolic_bp",
        "respiratory_rate",
        "oxygen_saturation",
        "temperature",
    ]
].describe()

,heart_rate,systolic_bp,diastolic_bp,respiratory_rate,oxygen_saturation,temperature
count,130635.000000,129039.000000,129037.000000,114227.000000,130755.000000,95704.000000
mean,68.462816,117.899372,66.938587,12.171409,99.575129,35.232957
std,13.855939,24.348045,13.123803,2.822158,1.025661,2.651873
min,46.000000,82.000000,44.500000,3.500000,85.000000,20.600000
25%,58.000000,98.500000,57.000000,9.500000,99.000000,35.500000
50%,66.000000,112.000000,66.000000,12.000000,100.000000,35.800000
75%,78.000000,135.000000,76.000000,14.000000,100.000000,36.200000
max,108.000000,178.000000,98.000000,20.000000,100.000000,37.000000


In [28]:
hb = labs[labs["item_name"] == "hb"].copy()

In [29]:
hb.head()

,subject_id,chart_time,item_name,value
85,114166470,3230320,hb,9.5
96,114166470,3231055,hb,10.4
118,114166470,4271435,hb,10.1
156,114166470,4272885,hb,9.5
170,114166470,4274340,hb,9.5


In [30]:
hb.info()

<class 'pandas.core.frame.DataFrame'>
Index: 811208 entries, 85 to 19464278
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   subject_id  811208 non-null  int64  
 1   chart_time  811208 non-null  int64  
 2   item_name   811208 non-null  object 
 3   value       811208 non-null  float64
dtypes: float64(1), int64(2), object(1)
memory usage: 30.9+ MB


In [31]:
hb_merge = (
    df[
        [
            "op_id",
            "subject_id",
            "opstart_time"
        ]
    ]
    .merge(
        hb,
        on="subject_id",
        how="left"
    )
)

In [32]:
hb_merge = hb_merge[
    hb_merge["chart_time"] <= hb_merge["opstart_time"]
]

In [33]:
hb_latest = (
    hb_merge
    .sort_values("chart_time")
    .groupby("op_id")
    .last()
    [["value"]]
    .rename(columns={"value": "hemoglobin"})
    .reset_index()
)

In [34]:
df = df.merge(
    hb_latest,
    on="op_id",
    how="left"
)

In [35]:
df["hemoglobin"].describe()

count    121768.000000
mean         13.017537
std           1.731248
min           7.600000
25%          11.900000
50%          13.200000
75%          14.200000
max          15.600000
Name: hemoglobin, dtype: float64

In [36]:
df["hemoglobin"].isna().mean()

np.float64(0.07018937080024434)

In [39]:
def add_latest_lab(df, labs_df, lab_name, output_column):
    """
    Add the latest laboratory result before surgery.
    """

    # Keep only one lab
    lab = labs_df[labs_df["item_name"] == lab_name].copy()

    # Match surgeries to this patient's labs
    lab = (
        df[
            [
                "op_id",
                "subject_id",
                "opstart_time"
            ]
        ]
        .merge(
            lab,
            on="subject_id",
            how="left"
        )
    )

    # Keep only labs before surgery
    lab = lab[
        lab["chart_time"] <= lab["opstart_time"]
    ]

    # Keep latest lab
    lab = (
        lab.sort_values("chart_time")
           .groupby("op_id")
           .last()[["value"]]
           .rename(columns={"value": output_column})
           .reset_index()
    )

    # Merge back
    return df.merge(
        lab,
        on="op_id",
        how="left"
    )

In [40]:
df = add_latest_lab(df, labs, "wbc", "wbc")
df = add_latest_lab(df, labs, "platelet", "platelets")
df = add_latest_lab(df, labs, "creatinine", "creatinine")
df = add_latest_lab(df, labs, "glucose", "glucose")

# Optional labs
df = add_latest_lab(df, labs, "sodium", "sodium")
df = add_latest_lab(df, labs, "potassium", "potassium")
df = add_latest_lab(df, labs, "albumin", "albumin")
df = add_latest_lab(df, labs, "ptinr", "inr")

In [41]:
df[
    [
        "hemoglobin",
        "wbc",
        "platelets",
        "creatinine",
        "glucose",
        "sodium",
        "potassium",
        "albumin",
        "inr"
    ]
].describe()

,hemoglobin,wbc,platelets,creatinine,glucose,sodium,potassium,albumin,inr
count,121768.000000,121339.000000,121209.000000,129724.000000,118169.000000,122231.000000,122235.000000,120744.000000,120339.000000
mean,13.017537,6.668218,238.536577,0.891712,109.323325,138.635150,4.009609,4.093569,0.998007
std,1.731248,2.362022,69.730096,0.604129,28.640126,3.067173,0.486139,0.459434,0.123968
min,7.600000,2.630000,44.000000,0.410000,78.000000,128.000000,3.000000,2.300000,0.880000
25%,11.900000,5.200000,189.000000,0.680000,91.000000,137.000000,3.600000,4.000000,0.930000
50%,13.200000,6.300000,233.000000,0.770000,101.000000,139.000000,4.000000,4.200000,0.980000
75%,14.200000,7.450000,273.000000,0.920000,115.000000,141.000000,4.400000,4.400000,1.030000
max,15.600000,19.440000,470.000000,5.540000,302.000000,147.000000,5.300000,4.700000,2.560000


In [42]:
(
    df[
        [
            "hemoglobin",
            "wbc",
            "platelets",
            "creatinine",
            "glucose",
            "sodium",
            "potassium",
            "albumin",
            "inr"
        ]
    ]
    .isna()
    .mean()
    .sort_values()
)

creatinine    0.009438
potassium     0.066623
sodium        0.066654
hemoglobin    0.070189
wbc           0.073465
platelets     0.074458
albumin       0.078009
inr           0.081101
glucose       0.097671
dtype: float64

In [43]:
df.to_parquet("../data/interim/features_after_labs.parquet", index=False)

In [44]:
diagnosis.columns.tolist()

['subject_id', 'chart_time', 'icd10_cm']

In [45]:
diagnosis.head(10)

,subject_id,chart_time,icd10_cm
0,190852492,325440,R06
1,190852492,325440,G20
2,142367193,0,I61
3,178346414,80640,A16
4,178346414,80640,J98
5,178346414,80640,R55
6,128748700,1440,R35
7,109644621,-1440,S02
8,172098362,-156960,T84
9,172098362,-41760,S73


In [46]:
diagnosis["icd10_cm"].nunique()

1136

In [47]:
diagnosis_clean = diagnosis.copy()

diagnosis_clean["icd10_cm"] = (
    diagnosis_clean["icd10_cm"]
    .astype("string")
    .str.upper()
    .str.strip()
    .str.replace(".", "", regex=False)
)

In [48]:
code = diagnosis_clean["icd10_cm"]

diagnosis_clean["diabetes"] = code.str[:3].isin(
    ["E10", "E11", "E12", "E13", "E14"]
)

diagnosis_clean["hypertension"] = code.str[:3].isin(
    ["I10", "I11", "I12", "I13", "I15"]
)

diagnosis_clean["ckd"] = code.str.startswith("N18")

diagnosis_clean["stroke"] = code.str[:3].between("I60", "I69")

diagnosis_clean["heart_failure"] = code.str.startswith("I50")

diagnosis_clean["copd"] = code.str.startswith("J44")

diagnosis_clean["cancer"] = code.str.startswith("C")

In [49]:
history_columns = [
    "diabetes",
    "hypertension",
    "ckd",
    "stroke",
    "heart_failure",
    "copd",
    "cancer",
]

diagnosis_clean[history_columns] = (
    diagnosis_clean[history_columns].astype("int8")
)

In [50]:
diagnosis_relevant = diagnosis_clean[
    diagnosis_clean[history_columns].any(axis=1)
].copy()

diagnosis_relevant.shape

(901194, 10)

In [51]:
diagnosis_by_surgery = (
    df[
        [
            "op_id",
            "subject_id",
            "opstart_time",
        ]
    ]
    .merge(
        diagnosis_relevant[
            [
                "subject_id",
                "chart_time",
                *history_columns,
            ]
        ],
        on="subject_id",
        how="left",
    )
)

In [52]:
diagnosis_by_surgery = diagnosis_by_surgery[
    diagnosis_by_surgery["chart_time"].isna()
    | (
        diagnosis_by_surgery["chart_time"]
        <= diagnosis_by_surgery["opstart_time"]
    )
]

In [53]:
history_by_surgery = (
    diagnosis_by_surgery
    .groupby("op_id", as_index=False)[history_columns]
    .max()
)

In [54]:
df = df.merge(
    history_by_surgery,
    on="op_id",
    how="left",
)

In [55]:
df[history_columns] = (
    df[history_columns]
    .fillna(0)
    .astype("int8")
)

In [56]:
history_summary = (
    df[history_columns]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .round(2)
)

history_summary

cancer           33.74
hypertension      7.89
diabetes          7.82
stroke            3.85
ckd               2.98
copd              1.21
heart_failure     0.78
dtype: float64

In [57]:
df[
    [
        "op_id",
        "subject_id",
        *history_columns,
    ]
].head(10)

,op_id,subject_id,diabetes,hypertension,ckd,stroke,heart_failure,copd,cancer
0,484069807,178742874,0,0,0,0,0,0,0
1,446270725,158995752,1,0,0,0,0,0,1
2,406892271,108553242,0,0,0,0,0,0,1
3,478413008,133278262,0,0,0,0,0,0,0
4,468516791,116924034,0,0,0,0,0,0,0
5,493866243,174229093,1,1,0,0,0,0,0
6,491416905,153073110,0,0,0,0,0,0,1
7,467806534,156776324,0,0,0,0,0,0,1
8,471265968,198640844,0,0,0,0,0,0,0
9,466411896,100259714,0,0,0,0,0,0,0


In [58]:
print("Rows:", len(df))
print("Unique surgeries:", df["op_id"].nunique())
print("Duplicate op_id:", df["op_id"].duplicated().sum())

Rows: 130960
Unique surgeries: 130960
Duplicate op_id: 0


In [59]:
OUTPUT_DIR = Path("../data/interim")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df.to_parquet(
    OUTPUT_DIR / "features_after_diagnoses.parquet",
    index=False,
)